<a href="https://colab.research.google.com/github/kshupe/wildlife-data-portfolio/blob/main/boundarybreach_krugerelephants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
# Elephant Movement Analysis: Kruger National Park
### Data Attribution & License
# **Study:** ThermochronTracking Elephants Kruger 2007
# **License:** CC BY-NC (Creative Commons Attribution-NonCommercial)
# **Principal Investigator:** Abi Tamim Vanak

# **Primary Citation:** > Slotow R, Thaker M, Vanak AT (2019) Data from: Fine-scale tracking of ambient temperature and movement reveals shuttling behavior of elephants to water. Movebank Data Repository. https://www.doi.org/10.5441/001/1.403h24q5

# **Associated Publication:** > Thaker M, Gupte PR, Prins HHT, Slotow R, Vanak AT. 2019. Fine-scale tracking of ambient temperature and movement reveals shuttling behavior of elephants to water. Front Ecol Evol. 7:4. https://doi.org/10.3389/fevo.2019.00004

# **Acknowledgements:** We thank SANParks for providing weather data and geographic shapefiles of Kruger NP, and the SANParks veterinary team for the collaring of elephants.

In [38]:
# 2. Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
# 3. Import the basics
import pandas as pd
import geopandas as gpd
import ecoscope
import os

print("Setup Complete! Your 'Field Office' is now live.")


Setup Complete! Your 'Field Office' is now live.


In [40]:
# Create file path for data
file_path = '/content/drive/MyDrive/Wildlife_Data/Projects/Boundary_Breach/Kruger_Elephants/Data/ThermochronTracking Elephants Kruger 2007.csv'

# Load the data
df = pd.read_csv(file_path)

# 2. Make timestamps readable (Standardizing time zones)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# 3. Create a GeoDataFrame
# This tells the computer that 'location-long' is X and 'location-lat' is Y
gdf = gpd.GeoDataFrame(df,
                       geometry=gpd.points_from_xy(df['location-long'], df['location-lat']),
                       crs=4326)

# 4. Convert to Ecoscope Relocations
relocs = ecoscope.base.Relocations.from_gdf(gdf,
                                            groupby_col='individual-local-identifier',
                                            time_col='timestamp')

print("Transformation Complete. Your data is now spatial")

Transformation Complete. Your data is now spatial


In [41]:
# Create a filter for 10 km/hr (a brisk pace for an elephant)
speed_filter = ecoscope.base.RelocsSpeedFilter(max_speed_kmhr=10.0)

# Apply the filter and remove 'junk' points
relocs.apply_reloc_filter(speed_filter, inplace=True)
relocs.remove_filtered(inplace=True)

print(f"Filtering finished. You have {len(relocs)} high-quality data points.")



Filtering finished. You have 283670 high-quality data points.


In [57]:
# 1. Print the 'Main' columns
print("--- MAIN COLUMNS ---")
print(relocs.columns.tolist())

# 3. See the first 2 rows to check the actual data values
print("\n--- DATA PREVIEW ---")
print(relocs.head(2))

--- MAIN COLUMNS ---
['extra__event-id', 'extra__visible', 'extra__timestamp', 'extra__location-long', 'extra__location-lat', 'extra__external-temperature', 'extra__sensor-type', 'extra__individual-taxon-canonical-name', 'extra__tag-local-identifier', 'extra__individual-local-identifier', 'extra__study-name', 'extra__utm-easting', 'extra__utm-northing', 'extra__utm-zone', 'extra__study-timezone', 'extra__study-local-timestamp', 'geometry', 'groupby_col', 'fixtime', 'junk_status']

--- DATA PREVIEW ---
   extra__event-id  extra__visible          extra__timestamp  \
0       9421351127            True 2007-08-13 00:30:00+00:00   
1       9421351128            True 2007-08-13 02:00:00+00:00   

   extra__location-long  extra__location-lat  extra__external-temperature  \
0              31.87091            -24.81373                         24.0   
1              31.87399            -24.81483                         23.0   

  extra__sensor-type extra__individual-taxon-canonical-name  \
0    